In [3]:
import pandas as pd
import numpy as np
from scipy.stats import kendalltau
from scipy.stats import chi2_contingency, random_table

file_path = "GenAI_Cleaned_Data.xlsx"

df = pd.read_excel(
    file_path,
    sheet_name="Users n103"
)

print("Sample size:", len(df))


Sample size: 103


In [5]:
#RQ2 - Workload reduction vs correction requirements
#Kendall's tau-b - 1 

workload_order = [
    "Disagree",
    "Neither agree nor disagree",
    "Agree"
]

correction_order = [
    "0–25%",
    "26–50%",
    "51–75%",
    "76–100%"
]

cross_tab = pd.crosstab(
    df["workload_reduction"],
    df["correction_range"]
).reindex(
    index=workload_order,
    columns=correction_order,
    fill_value=0
)

print(cross_tab)


workload_codes = {
    "Disagree": 1,
    "Neither agree nor disagree": 2,
    "Agree": 3
}

correction_codes = {
    "0–25%": 1,
    "26–50%": 2,
    "51–75%": 3,
    "76–100%": 4
}

x = df["workload_reduction"].map(workload_codes)
y = df["correction_range"].map(correction_codes)

assert x.notna().all() and y.notna().all()


tau, p_value = kendalltau(
    x,
    y,
    variant="b"
)

print("Kendall's tau-b:", round(tau, 3))
print("p-value:", round(p_value, 3))


correction_range            0–25%  26–50%  51–75%  76–100%
workload_reduction                                        
Disagree                        0       0       0        0
Neither agree nor disagree      4       4      17        6
Agree                          13      15      25       19
Kendall's tau-b: -0.038
p-value: 0.675


In [7]:
#RQ2 - Effort reduction vs correction requirements
#Kendall's tau-b - 2 

effort_order = [
    "Disagree",
    "Neither agree nor disagree",
    "Agree"
]

correction_order = [
    "0–25%",
    "26–50%",
    "51–75%",
    "76–100%"
]

cross_tab = pd.crosstab(
    df["effort_reduction"],
    df["correction_range"]
).reindex(
    index=effort_order,
    columns=correction_order,
    fill_value=0
)

print(cross_tab)


effort_codes = {
    "Disagree": 1,
    "Neither agree nor disagree": 2,
    "Agree": 3
}

correction_codes = {
    "0–25%": 1,
    "26–50%": 2,
    "51–75%": 3,
    "76–100%": 4
}

x = df["effort_reduction"].map(effort_codes)
y = df["correction_range"].map(correction_codes)

assert x.notna().all() and y.notna().all()


tau, p_value = kendalltau(
    x,
    y,
    variant="b"
)

print("Kendall's tau-b:", round(tau, 3))
print("p-value:", round(p_value, 3))


correction_range            0–25%  26–50%  51–75%  76–100%
effort_reduction                                          
Disagree                        0       0       0        3
Neither agree nor disagree      2       7      14        4
Agree                          15      12      28       18
Kendall's tau-b: -0.084
p-value: 0.349


In [9]:
#RQ3 - Main barrier vs perceived productivity 
#Chi-square and Monte Carlo test

barrier_order = [
    "Low trust in AI-generated output",
    "Lack of training",
    "Poor integration with my workflow",
    "No major challenges"
]

productivity_order = [
    "Disagree",
    "Neither agree nor disagree",
    "Agree"
]

table = pd.crosstab(
    df["main_barrier"],
    df["productivity"]
).reindex(
    index=barrier_order,
    columns=productivity_order,
    fill_value=0
)

print(table)


observed = table.to_numpy()

result = chi2_contingency(
    observed,
    correction=False
)

chi2 = result.statistic
expected = result.expected_freq
dof = result.dof
n = observed.sum()

cramers_v = np.sqrt(
    chi2 / (
        n * min(
            observed.shape[0] - 1,
            observed.shape[1] - 1
        )
    )
)

print("Chi-square:", round(chi2, 3))
print("Degrees of freedom:", dof)
print("Cells with expected count < 5:",
      np.sum(expected < 5))
print("Cramer's V:", round(cramers_v, 3))


rng = np.random.default_rng(20261107)

n_simulations = 200000
extreme_count = 0

for _ in range(20):
    simulated = random_table.rvs(
        observed.sum(axis=1),
        observed.sum(axis=0),
        size=10000,
        random_state=rng
    )

    simulated_chi2 = (
        (simulated - expected)**2 / expected
    ).sum(axis=(1, 2))

    extreme_count += np.count_nonzero(
        simulated_chi2 >= chi2 - 1e-12
    )

p_mc = (extreme_count + 1) / (n_simulations + 1)

print("Monte Carlo p-value:", round(p_mc, 4))


productivity                       Disagree  Neither agree nor disagree  Agree
main_barrier                                                                  
Low trust in AI-generated output          1                          15     20
Lack of training                          0                           1     27
Poor integration with my workflow         0                           6      9
No major challenges                       0                           2     22
Chi-square: 20.667
Degrees of freedom: 6
Cells with expected count < 5: 5
Cramer's V: 0.317
Monte Carlo p-value: 0.0007


In [11]:
#RQ4 - Task completion speed vs error reduction
#Kendall's tau-b  

response_order = [
    "Disagree",
    "Neither agree nor disagree",
    "Agree"
]

cross_tab = pd.crosstab(
    df["speed_agree"],
    df["error_reduction"]
).reindex(
    index=response_order,
    columns=response_order,
    fill_value=0
)

print(cross_tab)


response_codes = {
    "Disagree": 1,
    "Neither agree nor disagree": 2,
    "Agree": 3
}

speed = df["speed_agree"].map(response_codes)
errors = df["error_reduction"].map(response_codes)

assert speed.notna().all() and errors.notna().all()

tau, p_value = kendalltau(
    speed,
    errors,
    variant="b"
)

print("Kendall's tau-b:", round(tau, 3))
print("p-value:", round(p_value, 6))


error_reduction             Disagree  Neither agree nor disagree  Agree
speed_agree                                                            
Disagree                           0                           0      0
Neither agree nor disagree         2                          15      8
Agree                              1                          18     59
Kendall's tau-b: 0.394
p-value: 5.5e-05


In [13]:
#RQ5 - Review frequency vs perceived productivity
#Kendall's tau-b - 1

review_order = [
    "Never",
    "Sometimes",
    "Often"
]

productivity_order = [
    "Disagree",
    "Neither agree nor disagree",
    "Agree"
]

cross_tab = pd.crosstab(
    df["review_frequency"],
    df["productivity"]
).reindex(
    index=review_order,
    columns=productivity_order,
    fill_value=0
)

print(cross_tab)
print(cross_tab.sum(axis=1))


review_codes = {
    "Never": 1,
    "Sometimes": 2,
    "Often": 3
}

productivity_codes = {
    "Disagree": 1,
    "Neither agree nor disagree": 2,
    "Agree": 3
}

review = df["review_frequency"].map(review_codes)
productivity = df["productivity"].map(productivity_codes)

assert review.notna().all()
assert productivity.notna().all()

tau, p_value = kendalltau(
    review,
    productivity,
    variant="b"
)

print("Kendall's tau-b:", round(tau, 3))
print("p-value:", round(p_value, 3))


productivity      Disagree  Neither agree nor disagree  Agree
review_frequency                                             
Never                    0                           0      1
Sometimes                1                          14     41
Often                    0                          10     36
review_frequency
Never         1
Sometimes    56
Often        46
dtype: int64
Kendall's tau-b: 0.05
p-value: 0.61


In [15]:
#RQ5 - Correction requirements vs perceived productivity
#Kendall's tau-b - 2

correction_order = [
    "0–25%",
    "26–50%",
    "51–75%",
    "76–100%"
]

productivity_order = [
    "Disagree",
    "Neither agree nor disagree",
    "Agree"
]

cross_tab = pd.crosstab(
    df["correction_range"],
    df["productivity"]
).reindex(
    index=correction_order,
    columns=productivity_order,
    fill_value=0
)

print(cross_tab)


correction_codes = {
    "0–25%": 1,
    "26–50%": 2,
    "51–75%": 3,
    "76–100%": 4
}

productivity_codes = {
    "Disagree": 1,
    "Neither agree nor disagree": 2,
    "Agree": 3
}

correction = df["correction_range"].map(correction_codes)
productivity = df["productivity"].map(productivity_codes)

assert correction.notna().all()
assert productivity.notna().all()

tau, p_value = kendalltau(
    correction,
    productivity,
    variant="b"
)

print("Kendall's tau-b:", round(tau, 3))
print("p-value:", round(p_value, 3))


productivity      Disagree  Neither agree nor disagree  Agree
correction_range                                             
0–25%                    0                           2     15
26–50%                   0                           6     13
51–75%                   1                          11     30
76–100%                  0                           5     20
Kendall's tau-b: -0.019
p-value: 0.831


In [17]:
#RQ5 - Importance of human involvement vs perceived productivity
#Kendall's tau-b - 3

importance_order = [
    "Not important",
    "Moderately important",
    "Very important"
]

productivity_order = [
    "Disagree",
    "Neither agree nor disagree",
    "Agree"
]

cross_tab = pd.crosstab(
    df["human_importance"],
    df["productivity"]
).reindex(
    index=importance_order,
    columns=productivity_order,
    fill_value=0
)

print(cross_tab)


importance_codes = {
    "Not important": 1,
    "Moderately important": 2,
    "Very important": 3
}

productivity_codes = {
    "Disagree": 1,
    "Neither agree nor disagree": 2,
    "Agree": 3
}

importance = df["human_importance"].map(importance_codes)
productivity = df["productivity"].map(productivity_codes)

assert importance.notna().all()
assert productivity.notna().all()

tau, p_value = kendalltau(
    importance,
    productivity,
    variant="b"
)

print("Kendall's tau-b:", round(tau, 3))
print("p-value:", round(p_value, 3))


productivity          Disagree  Neither agree nor disagree  Agree
human_importance                                                 
Not important                0                           0      2
Moderately important         0                          14     30
Very important               1                          10     46
Kendall's tau-b: 0.108
p-value: 0.268
